# Naive Fine-tuning: Evaluación Class-IL

Este notebook entrena un baseline Naive en el escenario **Class-IL**. 
Aquí el modelo tiene una única cabeza de salida y debe decidir entre todas las clases vistas hasta ahora sin saber el ID de la tarea.

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader
from models import CNN, ClassIncrementalClassifier
from dataloaders import SequentialCIFAR10
from utils_class_il import evaluate_class_il

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 10
print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


In [4]:
seq_cifar = SequentialCIFAR10(batch_size=BATCH_SIZE)
backbone = CNN(in_channels=3, embedding_dim=32)
model = ClassIncrementalClassifier(backbone, embedding_dim=32, total_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for task_id in range(5):
    print(f"\n--- ENTRENANDO TAREA {task_id} ---")
    
    # 1. Activar nuevas clases en el modelo
    current_classes = seq_cifar.task_classes[task_id]
    model.add_task(current_classes)
    
    # 2. Obtener datos de la tarea actual
    train_ds = seq_cifar.get_task_train_dataset(task_id, remap_labels=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    # 3. Entrenamiento Naive
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f}")
    
    # 4. Evaluación Class-IL (todas las clases vistas hasta ahora)
    all_test = ConcatDataset([seq_cifar.get_task_test_dataset(tid) for tid in range(task_id + 1)])
    combined_loader = DataLoader(all_test, batch_size=BATCH_SIZE, shuffle=False)
    
    acc_class_il = evaluate_class_il(model, combined_loader, device, [])
    print(f"Precisión Class-IL tras Tarea {task_id}: {acc_class_il:.2f}%")


--- ENTRENANDO TAREA 0 ---
  Epoch 1/10 | Loss: 0.4753
  Epoch 2/10 | Loss: 0.3766
  Epoch 3/10 | Loss: 0.3337
  Epoch 4/10 | Loss: 0.3029
  Epoch 5/10 | Loss: 0.2898
  Epoch 6/10 | Loss: 0.2722
  Epoch 7/10 | Loss: 0.2710
  Epoch 8/10 | Loss: 0.2460
  Epoch 9/10 | Loss: 0.2626
  Epoch 10/10 | Loss: 0.2421
Precisión Class-IL tras Tarea 0: 92.55%

--- ENTRENANDO TAREA 1 ---
  Epoch 1/10 | Loss: 0.6644
  Epoch 2/10 | Loss: 0.5335
  Epoch 3/10 | Loss: 0.5022
  Epoch 4/10 | Loss: 0.5014
  Epoch 5/10 | Loss: 0.4695
  Epoch 6/10 | Loss: 0.4576
  Epoch 7/10 | Loss: 0.4492
  Epoch 8/10 | Loss: 0.4533
  Epoch 9/10 | Loss: 0.4419
  Epoch 10/10 | Loss: 0.4353
Precisión Class-IL tras Tarea 1: 40.88%

--- ENTRENANDO TAREA 2 ---
  Epoch 1/10 | Loss: 0.8542
  Epoch 2/10 | Loss: 0.4979
  Epoch 3/10 | Loss: 0.4497
  Epoch 4/10 | Loss: 0.4128
  Epoch 5/10 | Loss: 0.4013
  Epoch 6/10 | Loss: 0.3786
  Epoch 7/10 | Loss: 0.3783
  Epoch 8/10 | Loss: 0.3764
  Epoch 9/10 | Loss: 0.3540
  Epoch 10/10 | Loss: 